<a href="https://colab.research.google.com/github/eboekenh/Agentic-AI-Bootcamp/blob/main/Week_6_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-google-genai
!pip install langchain_community
!pip install langchain_core
!pip install pypdf
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [2]:
import os

GOOGLE_API_KEY = "your api key here"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

chat = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

result = chat.invoke("who is roger federer, answer in 2 lines")

In [10]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

print("Available models:")
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

Available models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/

In [8]:
print(result)

content="Roger Federer is a retired Swiss professional tennis player. He is widely regarded as one of the greatest tennis players of all time, having won 20 Grand Slam men's singles titles during his illustrious career." additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e1254-e683-7903-9882-7c30a8058b67-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 13, 'output_tokens': 214, 'total_tokens': 227, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 172}}


In [9]:
for m in chat.stream("who is roger federer, answer in 2 lines"):
  print(m.content, end = "", flush=True)

Roger Federer is a retired Swiss professional tennis player, widely regarded as one of the greatest of all time. He holds the record for 20 Grand Slam men's singles titles and is known for his elegant style and sportsmanship.

## PDF Based Chatbot

In [11]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Research Paper - Final.pdf")
documents = loader.load()

context = "\n\n".join(doc.page_content for doc in documents)

In [12]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables = ["context", "question"],
    template="""
    You are an intelligent document assistant. Your only knowledge comes from the provided document excerpts below.
Answer the user's question based **only** on the information contained in the following context.
If the answer is not clearly contained in the context, say:
"I don't have enough information in the document to answer this question accurately."

Context:
{context}

Question: {question}

Answer in a clear, concise and helpful way. Be direct — avoid unnecessary introductions like "According to the document" unless it adds clarity.
"""
)

In [13]:
formatted_prompt = prompt.format(
    context = context,
    question = "Who is roger federer"
)

In [14]:
result = chat.invoke(formatted_prompt)

print(result.content)

I don't have enough information in the document to answer this question accurately.


## Directory Based Chatbot


In [15]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

loader = DirectoryLoader(
    path = "/content/test",
    glob = "**/*.pdf",
    loader_cls = PyPDFLoader
)
documents = loader.load()

context = "\n\n".join(doc.page_content for doc in documents)

FileNotFoundError: Directory not found: '/content/test'

In [ ]:
formatted_prompt = prompt.format(
    context = context,
    question = "Adaboost selects a training subset randomly - is this correct? Just answer yes or no"
)

In [ ]:
result = chat.invoke(formatted_prompt)

print(result.content)